In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
df_bronze_orders = spark.read.table("de.p1.bronze_orders")

df_quarantine_orders = df_bronze_orders.filter(
    (F.col('order_id').isNull()) |
    (F.col('user_id') <= 0) |
    (F.col('product_id') <= 0) |
    (F.col('order_status').isin('CANCELLED', 'REFUNDED'))
)

df_quarantine_orders.withColumn('quarantine_reason',
                                F.when(F.col('order_id').isNull(), 'Missing Order ID')
                                .when((F.col('user_id') <= 0) | (F.col('product_id') <= 0), 'Invalid IDs')
                                .otherwise('Order Cancelled or Refunded')
                                ).write.format('delta').mode('overwrite').saveAsTable('de.p1.quarantine_orders')

In [0]:
df_silver_orders = df_bronze_orders.filter(
    (F.col('order_id').isNotNull()) &
    (F.col('user_id') > 0) &
    (F.col('product_id') > 0) &
    (~F.col('order_status').isin('CANCELLED', 'REFUNDED'))
).select(
    'order_id',
    'user_id',
    'product_id',
    'quantity',
    F.upper(F.col('order_status')).alias('order_status'),
    'order_timestamp',
    'source_file_date',
    F.current_timestamp().alias('silver_processed_at')
)

df_silver_orders.write.format('delta').mode('overwrite').saveAsTable('de.p1.silver_orders')
print('Data cleaning on Orders table has been done successfully. All the normal data have been written into silver_orders and abnormal data have been written into quarantine_orders.')

Data cleaning on Orders table has been done successfully. All the normal data have been written into silver_orders and abnormal data have been written into quarantine_orders.


In [0]:
df_current_users = spark.read.table('de.p1.bronze_users') \
    .select(
        'user_id',
        'tier',
        'city',
        'updated_at'
    )
target_table_name = 'de.p1.silver_users_scd2'

In [0]:
if not spark.catalog.tableExists(target_table_name):
    print('Initialitng the SCD Type 2 users table ...')
    df_initial_scd2 = df_current_users \
        .withColumn('start_date', F.col('updated_at')) \
        .withColumn('end_date', F.lit('9999-12-31 23:59:59')) \
        .withColumn('is_current', F.lit(True))
    df_initial_scd2.write.format('delta').mode('overwrite').saveAsTable(target_table_name)
else:
    print('Updating the SCD Type 2 users table ...')
    
    deltaTable = DeltaTable.forName(spark, target_table_name)
    deltaTable.alias('t').merge(
        df_current_users.alias('s'),
        't.user_id = s.user_id and t.is_current = true'
    ).whenMatchedUpdate(
        condition = 't.tier <> s.tier or t.city <> s.city',
        set = {
            'end_date': 's.updated_at',
            'is_current': 'false'
        }
    ).whenNotMatchedInsert(
        values = {
            'user_id': 's.user_id',
            'tier': 's.tier',
            'city': 's.city',
            'updated_at': 's.updated_at',
            'start_date': 's.updated_at',
            'end_date': '"9999-12-31 23:59:59"',
            'is_current': 'true'
        }
    ).execute()

print('All executions of silve layer have been done!')

Updating the SCD Type 2 users table ...
All executions of silve layer have been done!


In [0]:
display(spark.sql('select "Silver Orders" as tbl, count(*) as cnt from de.p1.silver_orders union all '
                  'select "Quarantine Orders", count(*) from de.p1.quarantine_orders union all '
                  'select "Silver Users SCD2", count(*) from de.p1.silver_users_scd2'))

tbl,cnt
Silver Orders,287
Quarantine Orders,313
Silver Users SCD2,100
